# Домашнее задание: Полноценная мультиагентная система с инструментами

Цель: спроектировать и реализовать систему из нескольких LLM‑агентов, которые координируются, обмениваются сообщениями и вызывают инструменты для решения реальной задачи end‑to‑end.


## Требования (обязательные)
- Роли агентов (минимум 3) с чётко разделёнными обязанностями и разными наборами инструментов для каждой роли.
- Коммуникация: шина сообщений или адресные сообщения (ask_agent / reply) с логами входящих/исходящих сообщений.
- Инструменты (минимум 3) с реальными побочными эффектами: например, работа с файлами/кодом/HTTP/данными/оценкой.
- SGR/Structured Output: схемы действий/сообщений (UseTool/AskAgent/Reply/Finish) с валидацией.
- Оркестрация: централизованный планировщик или децентрализованная логика (обосновать выбор).
- Демонстрация: один целевой сценарий, доведённый до finish, с логами и артефактами (код/отчёт/файлы).


## Оценивание (10 баллов)
- **3 балла — Дизайн агентов и протокола**
  - Чёткие роли и разграничение ответственности (1)
  - Разные доступные инструменты у разных ролей (1)
  - SGR/схемы и валидация structured output (1)
- **5 баллов — Мультиагентность и взаимодействие**
  - Координация и обмен сообщениями между агентами (2)
  - Реальные вызовы инструментов и побочные эффекты (2)
  - Обработка ошибок/ретраи/эскалация (1)
- **1 балл — Качество кода и отчёта**
  - Читаемость, структура, документация и воспроизводимость
- **1 балл — Бонус: >5 уникальных инструментов**
  - Разные по назначению функции, а не дубликаты


## Идеи задач
- Кодогенерация по спецификации: Planner → Coder → Tester → Reviewer.
- Data/RAG пайплайн: Ingestor → Indexer → Analyst → Evaluator.
- Веб‑интеграция: Researcher (web/http), Summarizer, Reporter, Verifier.

## Как сдавать
- Ноутбук с кодом агентов, инструментов и оркестратора.
- README с архитектурой (роли, взаимодействие), списком инструментов, схемами SGR, логами одного прогона, ограничениями и метриками.
- (Опционально) Короткое видео/гиф ≤3 мин с успешным прогоном.

## Штрафы
- − до 2: нет логов или невоспроизводимо
- − до 2: инструменты не делают реальных действий
- − до 1: смешение ролей/инструментов без обоснования


## Стартовые подсказки (не обязательно)
- Начните с 3 ролей и простого набора инструментов; затем добавляйте Reviewer/Verifier/Deployer.
- Схемы SGR: определите классы UseTool/AskAgent/Reply/Finish.
- Ограничьте инструменты по ролям на уровне оркестратора (enforcement).
- Логи: печатайте inbox/outbox, действие (JSON), результаты инструментов (stdout/error), diff при изменении файлов.
- Введите ретраи и сообщения «поясни ошибку», если JSON от модели невалиден или инструмент упал.


In [1]:
# 1) Настройка OpenRouter (обязательное использование LLM)

import os
from typing import Optional

os.environ["OPENROUTER_API_KEY"]="Your key here"

DEFAULT_MODEL = "qwen/qwen3-30b-a3b-instruct-2507"  # можно поменять на совместимую с JSON schema
openrouter_client = None

if os.environ.get("OPENROUTER_API_KEY"):
    try:
        from openai import OpenAI
        openrouter_client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ["OPENROUTER_API_KEY"],
        )
        print("OpenRouter готов. Модель:", DEFAULT_MODEL)
    except Exception as e:
        raise RuntimeError(f"Не удалось инициализировать OpenRouter: {e}")
else:
    raise RuntimeError("OPENROUTER_API_KEY не найден. Установите ключ в окружении: export OPENROUTER_API_KEY=...")


OpenRouter готов. Модель: qwen/qwen3-30b-a3b-instruct-2507


In [ ]:
# 2) вызов LLM с JSON Schema

from pydantic import BaseModel
from typing import Optional
import json

def call_llm_with_schema(
    prompt: str,
    response_schema: BaseModel,
    system_prompt: Optional[str] = None
) -> BaseModel:
    if openrouter_client is None:
        raise RuntimeError("OpenRouter клиент не инициализирован. Установите OPENROUTER_API_KEY и перезапустите kernel.")
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    schema_dict = response_schema.model_json_schema()
    resp = openrouter_client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": response_schema.__name__, "schema": schema_dict, "strict": True},
        },
        temperature=0.2,
        max_tokens=1500,
    )
    raw = resp.choices[0].message.content or ""
    
    def extract_json(s: str) -> str:
        s = s.strip()
        if s.startswith("```"):
            lines = s.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            s = "\n".join(lines)
        s = s.strip()
        try:
            json.loads(s)
            return s
        except Exception:
            pass
        first = s.find('{'); last = s.rfind('}')
        if first != -1 and last != -1 and last > first:
            cand = s[first:last+1]
            try:
                json.loads(cand)
                return cand
            except Exception:
                return s
        return s
    parsed = extract_json(raw)
    return response_schema.model_validate_json(parsed)


In [3]:
# 3) SGR: действия агента и сообщения

from typing import Union, Dict, Any, List, Literal
from pydantic import BaseModel, Field

class AskAgent(BaseModel):
    action: str = "ask_agent"
    target: Literal["planner", "coder", "tester", "reviewer", "researcher"]
    message: str

class UseTool(BaseModel):
    action: str = "use_tool"
    tool_name: str  # валидация будет на уровне оркестратора
    args: Dict[str, Any]

class Reply(BaseModel):
    action: str = "reply"
    content: str

class Finish(BaseModel):
    action: str = "finish"
    summary: str

class AgentAction(BaseModel):
    step: Union[AskAgent, UseTool, Reply, Finish]

class BusMessage(BaseModel):
    sender: str
    recipient: str  # конкретная роль/агент или "broadcast"
    content: str


In [ ]:
# 4) Workspace и инструменты (минимальный набор + шаблоны)

from io import StringIO
import sys
import traceback
import difflib

class Workspace:
    def __init__(self):
        self.files: Dict[str, str] = {}
        self.memory: Dict[str, Any] = {}

    # Базовые инструменты
    def store_code(self, filename: str, code: str) -> Dict[str, Any]:
        prev = self.files.get(filename, "")
        self.files[filename] = code
        diff = "\n".join(difflib.unified_diff(prev.splitlines(), code.splitlines(), fromfile=f"prev:{filename}", tofile=f"new:{filename}", lineterm=""))
        return {"message": f"stored {filename}", "filename": filename, "chars": len(code), "diff": diff[:2000]}

    def read_code(self, filename: str) -> Dict[str, Any]:
        return {"filename": filename, "code": self.files.get(filename, "")}

    def run_python(self, code: str) -> Dict[str, Any]:
        captured = StringIO()
        old = sys.stdout
        sys.stdout = captured
        out = {"stdout": "", "error": ""}
        try:
            allowed = {"print": print, "range": range, "len": len, "sum": sum, "min": min, "max": max, "abs": abs}
            exec_globals = {"__builtins__": allowed}
            exec_locals = self.memory
            exec(code, exec_globals, exec_locals)
            out["stdout"] = captured.getvalue()
        except Exception:
            out["error"] = traceback.format_exc()
        finally:
            sys.stdout = old
        return out

    def run_tests(self, filename: str, tests_code: str) -> Dict[str, Any]:
        code = self.files.get(filename, "")
        combined = code + "\n\n" + tests_code
        res = self.run_python(combined)
        return {"passed": res["error"] == "", **res}

# Регистр инструментов (измените под свои инструменты)
class Tools:
    def __init__(self, ws: Workspace):
        self.ws = ws

    def call(self, tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
        args = args or {}
        # Шаблоны под дополнительные инструменты (>5):
        if tool_name == "summarize_text":
            text = args.get("text", "")
            return {"summary": text[:200], "info": "stub: implement real summarizer or LLM call"}
        if tool_name == "transform_data":
            data = args.get("data", [])
            return {"size": len(data), "preview": str(data)[:200]}
        if tool_name == "plan_schedule":
            tasks = args.get("tasks", [])
            return {"plan": [f"step-{i+1}: {t}" for i, t in enumerate(tasks[:10])]} 
        return {"error": f"unknown tool {tool_name}"}

ws = Workspace()
tools = Tools(ws)



In [5]:
# 5) Шина сообщений и шаблон агента

from collections import deque

class MessageBus:
    def __init__(self):
        self.queue = deque()
        self.history: List[BusMessage] = []
    def send(self, msg: BusMessage):
        self.queue.append(msg); self.history.append(msg)
    def receive_for(self, recipient: str) -> List[BusMessage]:
        msgs = [m for m in list(self.queue) if m.recipient in (recipient, "broadcast")]
        for m in msgs:
            try:
                self.queue.remove(m)
            except ValueError:
                pass
        return msgs

bus = MessageBus()

class LlmAgent:
    def __init__(self, name: str, system_role: str, allowed_tools: List[str]):
        self.name = name
        self.system_role = system_role
        self.allowed_tools = allowed_tools
    
    def decide(self, goal: str) -> AgentAction:
        """Сформируйте промпт и позовите call_llm_with_schema. Верните AgentAction."""
        history_text = "\n".join([f"{m.sender}→{m.recipient}: {m.content}" for m in bus.history][-10:])
        tools_str = ", ".join(self.allowed_tools) if self.allowed_tools else "(нет)"
        prompt = f"""
Вы — {self.system_role}. Ваша роль: {self.name}.
Цель: {goal}
История:
{history_text}

Выберите одно действие:
- ask_agent (target, message)
- use_tool (tool_name in [{tools_str}], args)
- reply (content)
- finish (summary)

Верните только JSON по схеме AgentAction.
"""
        return call_llm_with_schema(prompt=prompt, response_schema=AgentAction, system_prompt=self.system_role)


In [6]:
# 6) Оркестратор (скелет)

class Orchestrator:
    def __init__(self):
        self.agents = {
            "planner":  LlmAgent("planner",  "Планировщик", []),
            "coder":    LlmAgent("coder",    "Разработчик", ["store_code", "read_code", "run_python"]),
            "tester":   LlmAgent("tester",   "Тестировщик", ["read_code", "run_tests"]),
            # добавляйте свои роли, например reviewer/researcher
        }
        self.allowed_tools = {name: a.allowed_tools for name, a in self.agents.items()}
    
    def step(self, goal: str, order: List[str]) -> bool:
        for name in order:
            inbox = bus.receive_for(name)
            if inbox:
                print(f"[{name}] inbox ({len(inbox)}):")
                for m in inbox[-3:]:
                    print(f"  {m.sender}→{m.recipient}: {m.content[:200]}")
            action = self.agents[name].decide(goal)
            s = action.step
            print(f"[{name}] action: {s}")
            if isinstance(s, AskAgent):
                bus.send(BusMessage(sender=name, recipient=s.target, content=s.message))
                print(f"[{name}] → [{s.target}] ask: {s.message}")
            elif isinstance(s, UseTool):
                if s.tool_name not in self.allowed_tools.get(name, []):
                    msg = f"роль {name} не имеет доступа к инструменту {s.tool_name}. Доступны: {self.allowed_tools.get(name, [])}"
                    print(f"[{name}] denied {s.tool_name}: {msg}")
                    bus.send(BusMessage(sender="orchestrator", recipient=name, content=msg))
                    continue
                result = tools.call(s.tool_name, s.args)
                print(f"[{name}] used {s.tool_name} → {str(result)[:300]}")
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"tool {s.tool_name} result: {result}"))
                if s.tool_name == "run_tests" and isinstance(result, dict) and result.get("passed") is True:
                    bus.send(BusMessage(sender="tester", recipient="broadcast", content="FINISH: tests passed"))
                    return True
            elif isinstance(s, Reply):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=s.content))
                print(f"[{name}] reply: {s.content}")
            elif isinstance(s, Finish):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"FINISH: {s.summary}"))
                print(f"[{name}] finish: {s.summary}")
                return True
        return False
    
    def run(self, goal: str, max_rounds: int = 8):
        print("=== ORCHESTRATION START ===")
        bus.send(BusMessage(sender="planner", recipient="broadcast", content=f"start: {goal}"))
        for r in range(1, max_rounds+1):
            print(f"\n--- ROUND {r} ---")
            if self.step(goal, ["planner", "coder", "tester"]):
                print("=== ORCHESTRATION FINISHED ===")
                break
        else:
            print("=== MAX ROUNDS REACHED ===")


In [ ]:
# 1) Задайте цель мультиагентной системы (строка)
# Примеры: "Сформировать отчёт по исследованию", "Спланировать эксперименты", "Проверить качество артефактов".
# goal = "ВАША_ЦЕЛЬ_ЗДЕСЬ"

# 2) (Опционально) Отправьте начальное сообщение в шину
# bus.send(BusMessage(sender="planner", recipient="broadcast", content=f"start: {goal}"))

# 3) (Опционально) При необходимости добавьте собственные инструменты в класс Tools
#    и расширьте allowed_tools для ролей в Orchestrator

# 4) Запустите оркестратор
# orch = Orchestrator()
# orch.run(goal=goal, max_rounds=8)

# 5) Выведите артефакты/логи из Workspace
# print("files:", list(ws.files.keys()))
# print("history:")
# for m in bus.history[-10:]:
#     print(f"{m.sender}→{m.recipient}: {m.content[:160]}")


### Необязательно использовать этот шаблон, можете написать свой!
